#  Modélisation - Data Challenge QRT

---
### Modèles testés
1. **A defaut** : Ridge and Laso Regression
2. **Cox Proportional Hazards** (modèle de survie classique)
3. **Random Survival Forest** (ensemble non-paramétrique)
4. **Gradient Boosting Survival** (sksurv)
5. **XGBoost** 
6. **LightGBM** 
###  Métrique
**C-index (Concordance Index)** - Mesure si les prédictions de risque sont dans le bon ordre

## 1. Imports et Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys
import random
from scipy.stats import rankdata
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw
from sksurv.util import Surv
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')
# Ajouter le dossier src au path
sys.path.append(str(Path('..') / 'src'))
from utils import (
    concordance_index_ipcw_score,
    cross_validate_survival,
    RegressionSurvivalWrapper,
    CoxWrapper,
    RSFWrapper,
    GBSurvivalWrapper,
    save_submission,
    save_model_results
)
# Sklearn
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
np.random.seed(42)
BASE_DIR = Path('..')

## 2. Chargement des Données Transformées

In [2]:
# Charger les données transformées
data = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'Train_transformed_data.csv')
print("="*60)
print(f"\nShape: {data.shape}")
# Séparer features et target
target_cols = ['ID', 'OS_YEARS', 'OS_STATUS']
feature_cols = [c for c in data.columns if c not in target_cols]
X = data[feature_cols].values
y_time = data['OS_YEARS'].values
y_event = data['OS_STATUS'].values.astype(bool)
ids = data['ID'].values
# Supprimer les lignes avec OS_YEARS manquant 
# Cela nous évite des erreurs de prédictions puisqu'on ne les connaît pas s'ils sont manquants.
valid_mask = ~np.isnan(y_time)
X = X[valid_mask]
y_time = y_time[valid_mask]
y_event = y_event[valid_mask]
ids = ids[valid_mask]
print(f"\nFeatures: {len(feature_cols)}")
print(" Target: OS_YEARS (temps), OS_STATUS (événement)")
print(f" Lignes conservées après filtrage des OS_YEARS manquants: {X.shape[0]}")
print("\nDistribution des événements:")
print(f" Censurés: {(~y_event).sum()} ({(~y_event).mean()*100:.1f}%)")
print(f" Décédés:  {y_event.sum()} ({y_event.mean()*100:.1f}%)")

# Créer la structure tableau pour sksurv
y_surv = np.array([(e, t) for e, t in zip(y_event, y_time)],
                  dtype=[('event', bool), ('time', float)])


Shape: (3323, 58)

Features: 55
 Target: OS_YEARS (temps), OS_STATUS (événement)
 Lignes conservées après filtrage des OS_YEARS manquants: 3173

Distribution des événements:
 Censurés: 1573 (49.6%)
 Décédés:  1600 (50.4%)


## 3. Validation croisée 

On utilise la fonction `cross_validate_survival` et les wrappers de modèles
importés depuis `utils.py` pour calculer l'IPCW-C-index 
avec une validation croisée en `KFold`.

## 4. Entraînement et Comparaison des Modèles

In [3]:
print("="*60)
print("ENTRAÎNEMENT DES MODÈLES")
# Dictionnaire pour stocker les résultats
results = {}
print("\n" + "-"*50)
print("MODÈLES DE RÉGRESSION")
print("-"*50)

regression_models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
}
for name, model in regression_models.items():
    print(f"\nEntraînement {name}...", end=" ")
    wrapped_model = RegressionSurvivalWrapper(model)
    res = cross_validate_survival(
        wrapped_model,
        X, y_time, y_event,
        n_splits=5, random_state=42,
        use_ipcw=True, tau=7
)
    results[name] = res
    print(f"IPCW-C-index: {res['mean']:.4f} (±{res['std']:.4f})")

ENTRAÎNEMENT DES MODÈLES

--------------------------------------------------
MODÈLES DE RÉGRESSION
--------------------------------------------------

Entraînement Ridge... IPCW-C-index: 0.6919 (±0.0113)

Entraînement Lasso... IPCW-C-index: 0.6953 (±0.0103)

Entraînement ElasticNet... IPCW-C-index: 0.6940 (±0.0108)

Entraînement RandomForest... IPCW-C-index: 0.6742 (±0.0087)

Entraînement GradientBoosting... IPCW-C-index: 0.6647 (±0.0051)


In [4]:
# ===============================================
# XGBOOST et LIGHTGBM
print("\n" + "-"*50)
print(" XGBOOST & LIGHTGBM")
print("-"*50)

print("\nEntraînement XGBoost...", end=" ")
xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_wrapped = RegressionSurvivalWrapper(xgb_model)
res = cross_validate_survival(xgb_wrapped, X, y_time, y_event, n_splits=5, random_state=42, use_ipcw=True, tau=7)
results['XGBoost'] = res
print(f"IPCW-C-index: {res['mean']:.4f} (±{res['std']:.4f})")

print("\nLightGBM...", end=" ")
lgb_model = lgb.LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
lgb_wrapped = RegressionSurvivalWrapper(lgb_model)
res = cross_validate_survival(lgb_wrapped, X, y_time, y_event, n_splits=5, random_state=42, use_ipcw=True, tau=7)
results['LightGBM'] = res
print(f"IPCW-C-index: {res['mean']:.4f} (±{res['std']:.4f})")


--------------------------------------------------
 XGBOOST & LIGHTGBM
--------------------------------------------------

Entraînement XGBoost... IPCW-C-index: 0.6576 (±0.0104)

LightGBM... IPCW-C-index: 0.6699 (±0.0062)


In [5]:
# =============================================================================
# MODÈLES DE SURVIE 
# =============================================================================
print("\n" + "-"*50)
print(" MODÈLES DE SURVIE (sksurv)")
print("-"*50)
# Cox Proportional Hazards
print(f"\nEntraînement CoxPH...", end=" ")
cox_model = CoxPHSurvivalAnalysis(alpha=0.1)
cox_wrapped = CoxWrapper(cox_model)
res = cross_validate_survival(
    cox_wrapped,
    X, y_time, y_event,
    n_splits=5, random_state=42,
    use_ipcw=True, tau=7
)
results['CoxPH'] = res
print(f"IPCW-C-index: {res['mean']:.4f} (±{res['std']:.4f})")
# Random Survival Forest
print(f"\nEntraînement Random Survival Forest...", end=" ")
rsf_model = RandomSurvivalForest(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)
rsf_wrapped = RSFWrapper(rsf_model)
res = cross_validate_survival(
    rsf_wrapped,
    X, y_time, y_event,
    n_splits=5, random_state=42,
    use_ipcw=True, tau=7
)
results['RSF'] = res
print(f"IPCW-C-index: {res['mean']:.4f} (±{res['std']:.4f})")
# Gradient Boosting Survival
print(f"\nEntraînement Gradient Boosting Survival...", end=" ")
gbs_model = GradientBoostingSurvivalAnalysis(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gbs_wrapped = GBSurvivalWrapper(gbs_model)
res = cross_validate_survival(
    gbs_wrapped,
    X, y_time, y_event,
    n_splits=5, random_state=42,
    use_ipcw=True, tau=7
)
results['GBSurvival'] = res
print(f"IPCW-C-index: {res['mean']:.4f} (±{res['std']:.4f})")


--------------------------------------------------
 MODÈLES DE SURVIE (sksurv)
--------------------------------------------------

Entraînement CoxPH... IPCW-C-index: 0.7033 (±0.0122)

Entraînement Random Survival Forest... IPCW-C-index: 0.7122 (±0.0085)

Entraînement Gradient Boosting Survival... IPCW-C-index: 0.7095 (±0.0096)


## 5. Comparaison des Résultats

In [6]:
print("\n" + "="*60)
print(" COMPARAISON DES MODÈLES")
# Créer DataFrame des résultats
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'C-index Mean': [results[m]['mean'] for m in results],
    'C-index Std': [results[m]['std'] for m in results]
}).sort_values('C-index Mean', ascending=False)

results_df['Rank'] = range(1, len(results_df) + 1)
results_df = results_df[['Rank', 'Model', 'C-index Mean', 'C-index Std']]
print("\n Classement par C-index:")
print(results_df.to_string(index=False))

# Meilleur modèle
best_model_name = results_df.iloc[0]['Model']
best_score = results_df.iloc[0]['C-index Mean']
print(f"\nMeilleur modèle: {best_model_name} (C-index = {best_score:.4f})")


 COMPARAISON DES MODÈLES

 Classement par C-index:
 Rank            Model  C-index Mean  C-index Std
    1              RSF      0.712241     0.008454
    2       GBSurvival      0.709469     0.009555
    3            CoxPH      0.703317     0.012200
    4            Lasso      0.695307     0.010256
    5       ElasticNet      0.693978     0.010808
    6            Ridge      0.691916     0.011315
    7     RandomForest      0.674152     0.008741
    8         LightGBM      0.669945     0.006164
    9 GradientBoosting      0.664716     0.005147
   10          XGBoost      0.657571     0.010404

Meilleur modèle: RSF (C-index = 0.7122)


In [7]:
import itertools
print("="*60)
print(" OPTIMISATION DES HYPERPARAMÈTRES (Top 3 modèles)")
print("="*60)
# Grilles simples d'hyperparamètres
param_grids = {
    'CoxPH': {
        'alpha': [0.01, 0.1, 0.5, 1.0],
    },
    'RSF': {
        'n_estimators': [100, 200],
        'max_depth': [8, 10],
        'min_samples_split': [5, 10],
    },
    'GBSurvival': {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.05, 0.1],
    },
}
best_params = {}
best_cv_scores = {}

for model_name, grid in param_grids.items():
    print(f"\n Modèle: {model_name}")
    keys = list(grid.keys())
    values_list = [grid[k] for k in keys]
    
    best_score = -np.inf
    best_cfg = None
    
    for values in itertools.product(*values_list):
        params = dict(zip(keys, values))
        
        if model_name == 'CoxPH':
            base_model = CoxPHSurvivalAnalysis(**params)
            wrapped = CoxWrapper(base_model)
        elif model_name == 'RSF':
            base_model = RandomSurvivalForest(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                min_samples_split=params['min_samples_split'],
                random_state=42,
                n_jobs=-1
)
            wrapped = RSFWrapper(base_model)
        elif model_name == 'GBSurvival':
            base_model = GradientBoostingSurvivalAnalysis(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                learning_rate=params['learning_rate'],
                random_state=42
            )
            wrapped = GBSurvivalWrapper(base_model)
        else:
            continue
        res = cross_validate_survival(
            wrapped,
            X, y_time, y_event,
            n_splits=5, random_state=42,
            use_ipcw=True, tau=7
)
        mean_score = res['mean']
        std_score = res['std']
        
        print(f"  {params}  IPCW-C-index = {mean_score:.4f} (±{std_score:.4f})")
        
        if mean_score > best_score:
            best_score = mean_score
            best_cfg = params
    
    best_params[model_name] = best_cfg
    best_cv_scores[model_name] = best_score
    print(f" Meilleurs paramètres pour {model_name}: {best_cfg} (IPCW-C-index = {best_score:.4f})")

print("\nLes meilleurs hyperparamètres:")
for name in ['RSF', 'GBSurvival', 'CoxPH']:
    if name in best_params:
        print(f"  {name}: {best_params[name]} (IPCW-C-index = {best_cv_scores[name]:.4f})")

 OPTIMISATION DES HYPERPARAMÈTRES (Top 3 modèles)

 Modèle: CoxPH
  {'alpha': 0.01}  IPCW-C-index = 0.7033 (±0.0122)
  {'alpha': 0.1}  IPCW-C-index = 0.7033 (±0.0122)
  {'alpha': 0.5}  IPCW-C-index = 0.7034 (±0.0122)
  {'alpha': 1.0}  IPCW-C-index = 0.7034 (±0.0122)
 Meilleurs paramètres pour CoxPH: {'alpha': 0.5} (IPCW-C-index = 0.7034)

 Modèle: RSF
  {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 5}  IPCW-C-index = 0.7086 (±0.0085)
  {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 10}  IPCW-C-index = 0.7109 (±0.0088)
  {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5}  IPCW-C-index = 0.7122 (±0.0086)
  {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 10}  IPCW-C-index = 0.7122 (±0.0085)
  {'n_estimators': 200, 'max_depth': 8, 'min_samples_split': 5}  IPCW-C-index = 0.7092 (±0.0090)
  {'n_estimators': 200, 'max_depth': 8, 'min_samples_split': 10}  IPCW-C-index = 0.7108 (±0.0080)
  {'n_estimators': 200, 'max_depth': 10, 'min_samples_split

## 6. Entraînement et Prédictions

In [8]:
print("="*60)
print(" ENTRAÎNEMENT DU MODÈLE FINAL")
print("="*60)
rsf_defaut = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 10,
    'random_state': 42,
    'n_jobs': -1,
}
gbs_defaut = {
    'n_estimators': 100,
    'max_depth': 5,
    'learning_rate': 0.1,
    'random_state': 42,
}
cox_default = {
    'alpha': 0.1,
}
# Mettre à jour avec les meilleurs paramètres 
if 'best_params' in globals():
    if 'RSF' in best_params and best_params['RSF'] is not None:
        rsf_defaut.update(best_params['RSF'])
    if 'GBSurvival' in best_params and best_params['GBSurvival'] is not None:
        gbs_defaut.update(best_params['GBSurvival'])
    if 'CoxPH' in best_params and best_params['CoxPH'] is not None:
        cox_default.update(best_params['CoxPH'])
final_models = {
    'CoxPH': CoxPHSurvivalAnalysis(**cox_default),
    'RSF': RandomSurvivalForest(**rsf_defaut),
    'GBSurvival': GradientBoostingSurvivalAnalysis(**gbs_defaut),
}
# Entraîner le meilleur modèle par validation croisée
best_model = final_models[best_model_name]
print(f"\n Modèle sélectionné: {best_model_name}")
# Fit sur toutes les données
if best_model_name in ['CoxPH', 'RSF', 'GBSurvival']:
    best_model.fit(X, y_surv)
    # Prédictions sur train 
    train_predictions = best_model.predict(X)
else:
    best_model.fit(X, y_time)
    train_predictions = -best_model.predict(X)  # Négatif pour le risque pour la regression
# Vérifier le C-index IPCW sur train 
train_cindex = concordance_index_ipcw_score(
    y_time, y_event,
    y_time, y_event,
    train_predictions, tau=7
)
print(f"\n IPCW-C-index sur train: {train_cindex:.4f}")

 ENTRAÎNEMENT DU MODÈLE FINAL

 Modèle sélectionné: RSF

 IPCW-C-index sur train: 0.7897


## 7. Amélioration des performancances de prédiction des trois modèles

1. **Combinaison des prédictions** des meilleurs modèles 
2. **Optimisation  des hyperparamètres**.


In [10]:
print("="*60)
print("  OPTIMISATION  DES HYPERPARAMÈTRES RSF")
# Grille 
rsf_param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [8, 10, 12, 15, None],
    'min_samples_split': [4, 6, 8, 10],
    'min_samples_leaf': [2, 3, 4, 5],
    'max_features': ['sqrt', 'log2', 0.3, 0.5],
}
print("\n Recherche randomisée")
best_rsf_score = 0
best_rsf_params = None
best_rsf_model = None
# Recherche manuelle avec CV 
np.random.seed(42)
n_iter = 30
for i in range(n_iter):
    random.seed(42 + i)  
    n_est = random.choice(rsf_param_dist['n_estimators'])
    max_d = random.choice(rsf_param_dist['max_depth'])
    min_split = random.choice(rsf_param_dist['min_samples_split'])
    min_leaf = random.choice(rsf_param_dist['min_samples_leaf'])
    max_feat = random.choice(rsf_param_dist['max_features'])
    params = {
        'n_estimators': n_est,
        'max_depth': max_d,
        'min_samples_split': min_split,
        'min_samples_leaf': min_leaf,
        'max_features': max_feat,
    }
    rsf_temp = RandomSurvivalForest(
        **params,
        random_state=42,
        n_jobs=-1
)
    # CV (3 folds) par validation croisée en se basant sur la comparaison d'IPCW moyen
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X):
        rsf_temp.fit(X[train_idx], y_surv[train_idx])
        pred = rsf_temp.predict(X[val_idx])
        score = concordance_index_ipcw_score(
            y_time[train_idx], y_event[train_idx],
            y_time[val_idx], y_event[val_idx],
            pred
        )
        scores.append(score)
    mean_score = np.mean(scores)
    if mean_score > best_rsf_score:
        best_rsf_score = mean_score
        best_rsf_params = params.copy()
        print(f"   [{i+1:2d}/{n_iter}] Nouveau meilleur: {mean_score:.4f} - {params}")
print(f"\n Meilleurs hyperparamètres RSF:")
for k, v in best_rsf_params.items():
    print(f"   {k}: {v}")
print(f"   C-index CV: {best_rsf_score:.4f}")
# Entraînement du modèle final optimisé
best_rsf_model = RandomSurvivalForest(**best_rsf_params, random_state=42, n_jobs=-1)
best_rsf_model.fit(X, y_surv)

  OPTIMISATION  DES HYPERPARAMÈTRES RSF

 Recherche randomisée


   [ 1/30] Nouveau meilleur: 0.7083 - {'n_estimators': 500, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2'}
   [ 2/30] Nouveau meilleur: 0.7096 - {'n_estimators': 200, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 0.3}
   [ 5/30] Nouveau meilleur: 0.7097 - {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'log2'}
   [ 7/30] Nouveau meilleur: 0.7109 - {'n_estimators': 500, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'log2'}
   [19/30] Nouveau meilleur: 0.7113 - {'n_estimators': 300, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'log2'}
   [30/30] Nouveau meilleur: 0.7116 - {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2'}

 Meilleurs hyperparamètres RSF:
   n_estimators: 300
   max_depth: None
   min_samples_split: 4
   min_samples_le

,n_estimators,300
,max_depth,None
,min_samples_split,4
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,'log2'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,42


In [11]:
print(f"best parameters:1s: {best_rsf_params}")

best parameters:1s: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2'}


In [13]:
print("="*60)
# Chargement des données test
test_data = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'Test_transformed_data.csv')
X_test = test_data[feature_cols].values
ids_test = test_data['ID'].values

print(f"\n Données test: {X_test.shape}")



#-------------------------------------
# Ré-entraînement total des 3 modèles sur toutes les données train
print("\n Entraînement final sur toutes les données...")
cox_final = CoxPHSurvivalAnalysis(alpha=0.1)
cox_final.fit(X, y_surv)
rsf_final = RandomSurvivalForest(**best_rsf_params, random_state=42, n_jobs=-1)
rsf_final.fit(X, y_surv)
gbs_final = GradientBoostingSurvivalAnalysis(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
gbs_final.fit(X, y_surv)
# Prédictions sur le test
pred_cox_test = cox_final.predict(X_test)
pred_rsf_test = rsf_final.predict(X_test)
pred_gbs_test = gbs_final.predict(X_test)
# Convertir en rangs va nous servir à renforcer l'importance relative de chaque patient
rank_cox_test = rankdata(pred_cox_test)
rank_rsf_test = rankdata(pred_rsf_test)
rank_gbs_test = rankdata(pred_gbs_test)
print(f"Exemple de valeurs de rang pour Cox: {rank_cox_test[:5]}")
# Pondération finale des modèles
best_weights = [0.7, 0.1, 0.1]   # Cox, RSF, GBS
risk_scores_ensemble = (
    best_weights[0] * rank_cox_test +
    best_weights[1] * rank_rsf_test +
    best_weights[2] * rank_gbs_test
)
# Sauvegarde des soumissions
output_dir = BASE_DIR / 'submissions'
output_dir.mkdir(parents=True, exist_ok=True)

save_submission(ids_test, risk_scores_ensemble, output_dir, filename='submission_combinaison_3_model2.csv')
risk_scores_rsf_opt = pred_rsf_test
save_submission(ids_test, risk_scores_rsf_opt, output_dir, filename='submission_rsf_optimized.csv')


 Données test: (1193, 55)

 Entraînement final sur toutes les données...
Exemple de valeurs de rang pour Cox: [1048.  740.  435.  991.  529.]
Soumission sauvegardée: ..\submissions\submission_combinaison_3_model2.csv
Soumission sauvegardée: ..\submissions\submission_rsf_optimized.csv


,ID,risk_score
0,KYW1,1044.944803
1,KYW2,937.854132
2,KYW3,530.872970
3,KYW4,1004.721999
4,KYW5,745.554110
...,...,...
1188,KYW1189,484.663372
1189,KYW1190,484.956521
1190,KYW1191,502.943938
1191,KYW1192,562.249279
